# ND2 Unified Discovery Notebook: Fase 3 (Legendre Generalizado)

Objetivo: Descubrir la ley universal $V \propto \frac{P_l(\cos \theta)}{r^{l+1}}$ para cualquier orden $l$.

In [ ]:
# 0. Sincronización Robusta (Colab/Kaggle)
import os
is_colab = os.path.exists('/content')
base_path = '/content' if is_colab else '/kaggle/working'
target_dir = os.path.join(base_path, 'ND2')
if not os.path.exists(target_dir):
    print(f"Clonando repositorio en {target_dir}...")
    os.system(f"git clone https://github.com/manramirezpi/ND2.git {target_dir}")
os.chdir(target_dir)
os.system("git pull origin main")
print(f"\n[OK] Trabajando en: {os.getcwd()}")

In [ ]:
# 1. Setup
!pip install gplearn setproctitle geopandas
!mkdir -p weights Multipolos/data/nd2_format Multipolos/results
if not os.path.exists('weights/checkpoint.pth'):
    !wget -O weights/checkpoint.pth https://github.com/yuzhTHU/ND2/releases/download/checkpoint.pth/checkpoint.pth

In [ ]:
# 2. GENERAR DATASET MULTI-L NORMALIZADO (Fase 3)
import numpy as np, json, os
def generate_multi_l_data_normalized(num_samples_per_l=500):
    all_r, all_theta, all_l, all_V = [], [], [], []
    for l in [1, 2, 3]:
        r = np.random.uniform(0.5, 6.0, num_samples_per_l)
        theta = np.random.uniform(0, np.pi, num_samples_per_l)
        if l == 1: P = np.cos(theta)
        elif l == 2: P = 0.5 * (3 * np.cos(theta)**2 - 1)
        elif l == 3: P = 0.5 * (5 * np.cos(theta)**3 - 3 * np.cos(theta))
        V = P / (r**(l+1))
        # Normalizar para que el AI vea todos los órdenes con el mismo peso
        std_V = np.std(V); V = V / std_V if std_V > 0 else V
        all_r.extend(r); all_theta.extend(theta); all_l.extend([float(l)] * num_samples_per_l); all_V.extend(V)
    data = {"V": 2, "E": 1, "A": [[0,1],[0,0]], "G": [[0,1]],
            "l_order": [[0.0, float(li)] for li in all_l],
            "theta": [[0.0, float(ti)] for ti in all_theta],
            "r_node": [[0.0, float(ri)] for ri in all_r],
            "target": [[0.0, float(vi)] for vi in all_V]}
    os.makedirs("Multipolos/data/nd2_format", exist_ok=True)
    json.dump(data, open("Multipolos/data/nd2_format/MULTIL_nd2.json", "w"))
    print(f"Dataset Multi-L NORMALIZADO generado.")
generate_multi_l_data_normalized()

## Fase 3.5: Diagnóstico de Recurrencia (Isolating Legendre)

Si la búsqueda global falla, realizamos una reducción natural generalizada: $V_{target} = V \cdot r^{l+1}$. 
Esto permite al AI enfocarse únicamente en descubrir la **Recurrencia de Legendre** (cómo cambia el polinomio angular con $l$).

In [ ]:
# 5. REDUCCIÓN NATURAL MULTI-L (Diagnóstico)
import json, numpy as np
with open("Multipolos/data/nd2_format/MULTIL_nd2.json", "r") as f: data = json.load(f)

target = np.array(data['target'])
r = np.array(data['r_node'])
l = np.array(data['l_order'])

# Limpiamos el radio usando la potencia l+1
# target = V * r**(l+1)
target[:, 1] = target[:, 1] * (r[:, 1]**(l[:, 1] + 1))

data['target'] = target.tolist()
with open("Multipolos/data/nd2_format/MULTIL_REDUCED_nd2.json", "w") as f: json.dump(data, f)
print("Dataset Reducido Multi-L generado. Objetivo: Descubrir P_l(theta).")

In [ ]:
# 6. BÚSQUEDA DE RECURRENCIA (Fase 3.5)
!python3 search.py \
    --data Multipolos/data/nd2_format/MULTIL_REDUCED_nd2.json \
    --vars l_order theta \
    --target_var target \
    --episodes 1000 \
    --device cuda

In [ ]:
# 7. AUTO-PUSH
def get_token_reloaded():
    try: from kaggle_secrets import UserSecretsClient; return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except: pass
    try: from google.colab import userdata; return userdata.get('GITHUB_TOKEN')
    except: pass
    return None
token = get_token_reloaded()
if token:
    os.system("git config --global user.email 'manuel@researcher.phys'")
    os.system("git config --global user.name 'ND2-Auto-Agent'")
    os.system(f"git remote set-url origin https://{token}@github.com/manramirezpi/ND2.git")
    !git add Multipolos/results/ Multipolos/data/nd2_format/MULTIL_nd2.json
    !git commit -m "Fase 3.5: Resultados de Recurrencia de Legendre"
    !git push origin main
    print("\n[OK] Resultados subidos.")
else: print("\n[!] GITHUB_TOKEN no encontrado.")